In [1]:
import pymysql
import requests

base_url = "https://rickandmortyapi.com/api/character"


# Extract Anime Cartoon Data from above REST API
def fetch_data(base_url):
    try:
        response = requests.get(base_url)
        if response.status_code == 200:
            meta_data = response.json()
            return meta_data
        else:
            print(f'Error: Unable to fetch data, status code {response.status_code}')
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")



In [2]:
# Establish database connection
connection = pymysql.connect(
    host='localhost',
    user='root',
    password='sr2910',
    db='spotify_db'
)

cursor = connection.cursor()

# Create table query
create_table_query = """ 
CREATE TABLE IF NOT EXISTS testing_api(
    id INT PRIMARY KEY,
    name VARCHAR(255),
    gender VARCHAR(50),
    species VARCHAR(50),
    status VARCHAR(50),
    episodes INT
);
"""
cursor.execute(create_table_query)
connection.commit()

In [3]:

# Transform the Data by creating a function
def extracted_data(jsondata):
    all_data = []
    for item in jsondata['results']:  # Accessing list in the dictionary by for loop
        extract_data = {
            'Id': item['id'],
            'Name': item['name'],
            'Gender': item['gender'],
            'Species': item['species'],
            'Status': item['status'],
            'Episodes': len(item['episode'])
        }
        all_data.append(extract_data)

        try:
            insert_query = """INSERT INTO testing_api (id, name, gender, species, status, episodes)
                              VALUES (%s, %s, %s, %s, %s, %s)                   
                              """
            cursor.execute(insert_query, (
                item['id'],
                item['name'],
                item['gender'],
                item['species'],
                item['status'],
                len(item['episode'])
            ))
        except pymysql.MySQLError as e:
            print(f"Error occurred: {e}")

    connection.commit()  # Commit all insertions after the loop
    return all_data




In [4]:
# Fetch data from API
jsondata = fetch_data(base_url)
print(f'Fetched JSON Data: {jsondata}')

# Process and insert data into the database
try:
    data = extracted_data(jsondata)
    print("Extracted Data:")
    for item in data:
        print(item)
except Exception as e:
    print(f"An error occurred during data processing: {e}")
finally:
    # Close resources
    if cursor:
        cursor.close()
    if connection:
        connection.close()


Fetched JSON Data: {'info': {'count': 826, 'pages': 42, 'next': 'https://rickandmortyapi.com/api/character?page=2', 'prev': None}, 'results': [{'id': 1, 'name': 'Rick Sanchez', 'status': 'Alive', 'species': 'Human', 'type': '', 'gender': 'Male', 'origin': {'name': 'Earth (C-137)', 'url': 'https://rickandmortyapi.com/api/location/1'}, 'location': {'name': 'Citadel of Ricks', 'url': 'https://rickandmortyapi.com/api/location/3'}, 'image': 'https://rickandmortyapi.com/api/character/avatar/1.jpeg', 'episode': ['https://rickandmortyapi.com/api/episode/1', 'https://rickandmortyapi.com/api/episode/2', 'https://rickandmortyapi.com/api/episode/3', 'https://rickandmortyapi.com/api/episode/4', 'https://rickandmortyapi.com/api/episode/5', 'https://rickandmortyapi.com/api/episode/6', 'https://rickandmortyapi.com/api/episode/7', 'https://rickandmortyapi.com/api/episode/8', 'https://rickandmortyapi.com/api/episode/9', 'https://rickandmortyapi.com/api/episode/10', 'https://rickandmortyapi.com/api/episo